# **Stage 1 - Import and Check Data**

This stage imports the required Python libraries and loads the original M5 datasets, including the calendar table, sales table, and sell price table. The purpose is to understand the structure, size, and key variables of each dataset before cleaning and transformation.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
plt.rcParams.update({"figure.dpi": 110, "font.size": 12})

DATA = Path(r"C:\Users\kyoaj\ALY 6140\ALY 6140\final project\m5")          # Original csv directory
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

### Calendar Table Overview

The calendar table contains 1,969 rows and 13 columns. It provides date-related information such as weekday, month, year, event names, event types, and SNAP indicators for different states. These variables are help to explain demand changes caused by seasonality, holidays, and external events.

In [2]:
# check calendar
calendar = pd.read_csv(DATA / "calendar.csv")
print("shape:", calendar.shape)
calendar.head()

shape: (1969, 13)


,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,NaN,NaN,NaN,NaN,1,0,1


In [3]:
# observations of calendar
print("event type:", calendar["event_type_1"].dropna().unique())
print(calendar.isna().sum()[lambda s: s > 0])

event type: <ArrowStringArray>
['Sporting', 'Cultural', 'National', 'Religious']
Length: 4, dtype: str
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
dtype: int64


### Sales Table Overview

The sales table contains 30,490 product-store time series and 1,941 daily sales columns. Each row represents one product sold in one store, while the columns `d_1` to `d_1941` represent daily unit sales over time.

In [4]:
# check sales_train_evaluation
sales = pd.read_csv(DATA / "sales_train_evaluation.csv")
print("shape:", sales.shape)
sales.head()

shape: (30490, 1946)


,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [5]:
# Distinguish between the id column and the date column
id_cols = [c for c in sales.columns if not c.startswith("d_")]
d_cols  = [c for c in sales.columns if c.startswith("d_")]
print(id_cols)
print(len(d_cols), "→", d_cols[0], "...", d_cols[-1])

['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
1941 → d_1 ... d_1941


In [6]:
# observations of sales
print(sales["cat_id"].value_counts())
foods_mask = sales["cat_id"] == "FOODS"
print(sales.loc[foods_mask, "dept_id"].value_counts())
print(sales.groupby("state_id")["store_id"].unique())

cat_id
FOODS        14370
HOUSEHOLD    10470
HOBBIES       5650
Name: count, dtype: int64
dept_id
FOODS_3    8230
FOODS_2    3980
FOODS_1    2160
Name: count, dtype: int64
state_id
CA    [CA_1, CA_2, CA_3, CA_4]
TX          [TX_1, TX_2, TX_3]
WI          [WI_1, WI_2, WI_3]
Name: store_id, dtype: object


### Sell Price Table Overview

The sell price table contains weekly product prices by item and store. The price range varies from 0.01 to 107.32, showing that products have different price levels.

In [7]:
# check sell_prices
prices = pd.read_csv(DATA / "sell_prices.csv")
print("shape:", prices.shape)
print(prices.head())

print("\nRange of Price:", prices["sell_price"].min(), "-", prices["sell_price"].max())

shape: (6841121, 4)
  store_id        item_id  wm_yr_wk  sell_price
0     CA_1  HOBBIES_1_001     11325        9.58
1     CA_1  HOBBIES_1_001     11326        9.58
2     CA_1  HOBBIES_1_001     11327        8.26
3     CA_1  HOBBIES_1_001     11328        8.26
4     CA_1  HOBBIES_1_001     11329        8.26

Range of Price: 0.01 - 107.32


### Create FOODS Dataset

Only FOODS items are selected from the full sales table. A unique `id` is created by combining item and store information. This identifier allows each product-store combination to be tracked as an individual time series.

In [8]:
# Creat foods table
foods = sales[sales["cat_id"] == "FOODS"].copy()

# Build a unique identifier
foods["id"] = foods["item_id"] + "_" + foods["store_id"] + "_evaluation"

print(len(foods))
print("example id:", foods["id"].iloc[0])
print("Counts of id:", foods["id"].nunique())

14370
example id: FOODS_1_001_CA_1_evaluation
Counts of id: 14370


### Convert Wide Data to Long Format

The original sales table is in wide format, with one column for each sales day. To make the dataset easier to merge, analyze, and model, the table is converted into long format. After transformation, each row represents one product-store observation on one specific day.

In [9]:
# Wide to Long Format
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
d_cols  = [c for c in foods.columns if c.startswith("d_")]

long = foods.melt(
    id_vars=id_cols,
    value_vars=d_cols,
    var_name="d",
    value_name="sales",
)

print("Before melt:", foods.shape, "wide")
print("After melt:", long.shape, "long")
print("Verify:", 14370 * 1941)
long.head()

Before melt: (14370, 1947) wide
After melt: (27892170, 8) long
Verify: 27892170


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,0
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,0
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,0
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3


### Create Day Number

The `d` column is converted into a numeric day index called `d_num`. This makes it possible to merge the sales data with the calendar table and align each sales record with its actual calendar date.

In [10]:
# Draw the date serial number from column d
long["d_num"] = long["d"].str.replace("d_", "", regex=False).astype("int32")

print(long[["id", "d", "d_num", "sales"]].head(3))
print("\nRange of d_num:", long["d_num"].min(), "-", long["d_num"].max())

                            id    d  d_num  sales
0  FOODS_1_001_CA_1_evaluation  d_1      1      3
1  FOODS_1_002_CA_1_evaluation  d_1      1      0
2  FOODS_1_003_CA_1_evaluation  d_1      1      0

Range of d_num: 1 - 1941


In [11]:
# Generate d_num for Calendar csv
calendar = calendar.reset_index(drop=True)
calendar["d_num"] = (calendar.index + 1).astype("int32")

print("calendar days:", len(calendar), "| sales days:", long["d_num"].max())
print(calendar[["d_num", "date", "weekday", "month", "year"]].head(3))

calendar days: 1969 | sales days: 1941
   d_num        date   weekday  month  year
0      1  2011-01-29  Saturday      1  2011
1      2  2011-01-30    Sunday      1  2011
2      3  2011-01-31    Monday      1  2011


### Merge Calendar Features

The calendar information is merged into the long sales table using `d_num`. After the merge, each sales record includes date, week, month, year, event, and SNAP-related features.

In [12]:
# Merge calendar to long
cal_cols = [c for c in calendar.columns if c != "weekday"]
calendar["date"] = pd.to_datetime(calendar["date"])

long = long.merge(calendar[cal_cols], on="d_num", how="left")

print(long.shape)
print(long["date"].min().date(), "→", long["date"].max().date())
long[["id", "date", "wday", "month", "event_name_1", "snap_CA"]].head(3)

(27892170, 21)
2011-01-29 → 2016-05-22


,id,date,wday,month,event_name_1,snap_CA
0,FOODS_1_001_CA_1_evaluation,2011-01-29,1,1,NaN,0
1,FOODS_1_002_CA_1_evaluation,2011-01-29,1,1,NaN,0
2,FOODS_1_003_CA_1_evaluation,2011-01-29,1,1,NaN,0


### Merge Sell Price Data

The sell price table is merged with the sales data using `store_id`, `item_id`, and `wm_yr_wk`. This adds weekly price information to each daily sales record.

In [13]:
# Merge sell_prices
prices = pd.read_csv(DATA / "sell_prices.csv")

long = long.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
)

print(long.shape)
print("sell_price missing proportion:", f"{long['sell_price'].isna().mean()*100:.1f}%")
long[["id", "date", "sales", "sell_price"]].head(3)

(27892170, 22)
sell_price missing proportion: 21.8%


,id,date,sales,sell_price
0,FOODS_1_001_CA_1_evaluation,2011-01-29,3,2.00
1,FOODS_1_002_CA_1_evaluation,2011-01-29,0,7.88
2,FOODS_1_003_CA_1_evaluation,2011-01-29,0,2.88


### Check Missing Sell Prices

Some sell prices are missing because certain products were not yet available in specific stores during the early period. These missing price records usually correspond to pre-launch periods where the product had zero sales.

In [14]:
# Hypothesis verification: The price of the product is missing before it is put on the shelves
# Which sequences have rows with missing prices
has_gap = long.groupby("id")["sell_price"].apply(lambda s: s.isna().any())
late_ids = has_gap[has_gap].index

print("The sequence numbers of the pre-listing stage:", len(late_ids), "/", long["id"].nunique())

# Take a look at one
sample_id = long[long["id"] == late_ids[0]].sort_values("d_num")
first_price_day = sample_id.loc[sample_id["sell_price"].notna(), "d_num"].min()
print("\nSeries:", late_ids[0])
print("The first price was at the", first_price_day, "days")

before = sample_id[sample_id["d_num"] < first_price_day]
print("Pre- launch period with missing price:", len(before), "| Total sales:", before["sales"].sum())

The sequence numbers of the pre-listing stage: 8923 / 14370

Series: FOODS_1_001_TX_3_evaluation
The first price was at the 8 days
Pre- launch period with missing price: 7 | Total sales: 0


### Remove Pre-Launch Records

Rows with missing sell prices are removed because they mostly represent periods before a product was launched in a store. Removing these pseudo-zero sales records helps avoid misleading the model and improves the quality of the final dataset.

In [15]:
# Remove rows before the product was launched
before_rows = len(long)
long = long[long["sell_price"].notna()].copy()
after_rows = len(long)

print(f"Rows before filtering: {before_rows:,}")
print(f"Rows after filtering: {after_rows:,}")
print(f"Removed {before_rows - after_rows:,} rows ({(before_rows-after_rows)/before_rows*100:.1f}%) of pre-launch pseudo-zeros")

Rows before filtering: 27,892,170
Rows after filtering: 21,798,313
Removed 6,093,857 rows (21.8%) of pre-launch pseudo-zeros


### Optimize Data Types

The dataset is very large, so data type optimization is applied to reduce memory usage. Categorical columns are converted to more efficient data types, and numeric columns are compressed where possible. This reduces memory usage from 5.33 GB to 1.11 GB, making the dataset easier to store and process.

In [16]:
# === Cell 19: Apply Data Type Optimization ===
from m5_utils import optimize_dtypes

print("Before optimization:")
print(f"  Memory usage: {long.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(long.dtypes)
# Compress data types to reduce memory footprint
long = optimize_dtypes(long)

print("\nAfter optimization:")
print(f"  Memory usage: {long.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print("\nOptimized dtypes:")
print(long.dtypes)

Before optimization:
  Memory usage: 5.33 GB
id                         str
item_id                    str
dept_id                    str
cat_id                     str
store_id                   str
state_id                   str
d                          str
sales                    int64
d_num                    int32
date            datetime64[us]
wm_yr_wk                 int64
wday                     int64
month                    int64
year                     int64
event_name_1               str
event_type_1               str
event_name_2               str
event_type_2               str
snap_CA                  int64
snap_TX                  int64
snap_WI                  int64
sell_price             float64
dtype: object

After optimization:
  Memory usage: 1.11 GB

Optimized dtypes:
id                    category
item_id               category
dept_id               category
cat_id                category
store_id              category
state_id              category
d        

### Save Cleaned Dataset

The cleaned FOODS dataset is saved as a Parquet file. Parquet is used because it preserves optimized data types, supports compression, and provides faster reading and writing performance than CSV. The final cleaned dataset contains 21,798,313 rows and 22 columns.

In [17]:
# Save Cleaned Long Table
# Use parquet format to preserve optimized dtypes, maximize compression, and speed up I/O
long.to_parquet(OUT / "foods_clean.parquet", index=False)

print(f"Successfully saved to:", OUT / "foods_clean.parquet")
print(f"Final data shape:", long.shape)
print(f"Available columns:", long.columns.tolist())

Successfully saved to: outputs\foods_clean.parquet
Final data shape: (21798313, 22)
Available columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'd_num', 'date', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']
